# Tutorial 14: End-to-End Coherent Scattering Experiment

This is the maintained replacement for the old monolithic scattering notebooks. It defines one complete magnetic Fourier-transform holography experiment, simulates matched CR and CL measurements, and compares charge-like sums with magnetic helicity differences at every stage.

The workflow is: illumination/source → experimental geometry → sample and mask → exit waves → ideal holograms → corrupted detector images → FTH reconstructions.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim
plt.rcParams["figure.constrained_layout.use"] = True


## 1. Define the illumination and X-ray source

Define the photon energy, flux, CR/CL states, transverse coherence length, and spatial beam parameters first. The coherence length belongs conceptually to the illumination even though it is stored on `XRayConfig` and applied when the ideal far field is projected onto the detector. Larger coherence lengths preserve finer interference fringes.


In [ ]:
# ########################################
# ILLUMINATION AND X-RAY SOURCE
# ########################################
xray_energy = 778.0                       # eV
photon_flux = 5e8                         # photons / second
polarizations = ("CR", "CL")
transverse_coherence_length = (100e-6, 100e-6)  # m, (y, x)

illumination_function = "gaussian"
illumination_center = np.array([0.0, 0.0])       # m, (y, x)
illumination_focus_distance = 0.0                # m from beam waist
illumination_fwhm = 60.0e-6                      # m at beam waist
illumination_alpha_beam = (0.0, 0.0)             # rad, (alpha_y, alpha_x)

xray = sim.XRayConfig(
    energy=xray_energy, photon_flux=photon_flux, pol=polarizations[0],
    coherence_length=transverse_coherence_length,
)
xray.setup()


## 2. Define detector, acquisition, corruption, and beamstop parameters

The detector geometry and source wavelength determine the sample-plane pixel size. Acquisition parameters control the frame stack. Detector-response parameters convert photons to counts and add readout effects. Photon-event kernels are a separate corruption mechanism: `sigma_photon=0` disables this spreading entirely, while smaller positive sigma and kernel size values produce sharper events. Beamstop `sigma` controls only the physical beamstop-edge transition and is not the photon-kernel width.

Hot/cold coordinates and per-pixel baseline values repeat for a `camera_seed`; temporal sigma controls make their values oscillate around those baselines. Cosmic rays are sampled per detection as short, blurred Gaussian ellipses with independently varying 2–3 aspect ratios and peak values. Missing exposure time means 1 second for cosmic-ray sampling.


In [ ]:
# ########################################
# DETECTOR GEOMETRY
# ########################################
detector_shape = (512, 512)
detector_pixel_size = 20e-6       # m
sample_detector_distance = 0.075  # m
detector_center = tuple(np.array(detector_shape) // 2)
oversampling = 2

# ########################################
# ACQUISITION
# ########################################
number_frames = 100
max_counts_per_frame = 64e3
exposure_time = 1.0               # s per frame

# ########################################
# DETECTOR RESPONSE
# ########################################
readout_noise_average = 20
readout_noise_sigma = 3
detector_threshold = 64e3
counts_per_photon = 180
quantum_efficiency = 0.9
detector_noise_seed = 12

# ########################################
# PHOTON-EVENT KERNELS
# ########################################
sigma_photon = 0.15                # pixels; set 0.0 for no event spreading
photon_kernel_size = 9             # pixels; odd integer
photon_n_classes = 1
photon_n_variants = 30
photon_irregularity = 2.0
regenerate_photon_kernels = False
photon_kernel_seed = 21

# CAMERA-PERSISTENT AND EXPOSURE-TRANSIENT ARTIFACTS
camera_seed = 2026
average_hot_pixels = 35.0
average_cold_pixels = 15.0
flicker_fraction = 0.2
flicker_probability = 0.5
hot_pixel_value = 55000.0
hot_pixel_value_spread = 0.08
hot_pixel_temporal_sigma = 0.03
cold_pixel_value = 150.0
cold_pixel_value_spread = 0.20
cold_pixel_temporal_sigma = 0.08
cosmic_rays_per_second = 0.2
cosmic_ray_value = 50000.0
cosmic_ray_value_spread = 0.15
cosmic_ray_length_range = (2.0, 5.0)
cosmic_ray_aspect_ratio_range = (2.0, 3.0)

# ########################################
# BEAMSTOP
# ########################################
beamstop_distance = 0.02          # m from detector
beamstop_radius = 0.12e-3            # m
beamstop_edge_sigma = 1e-6      # m; broad edge smoothing, independent of photon kernels
beamstop_wire_width = 10e-6   # m
beamstop_wire_bend = 15e-6         # m
beamstop_angle = np.deg2rad(20)

beamstop = sim.BeamstopConfig(
    bs_method="circular", bs_detector_distance=beamstop_distance,
    bs_center=detector_center,
    bs_config={
        "radius": beamstop_radius, "sigma": beamstop_edge_sigma,
        "wire_width": beamstop_wire_width, "wire_bend": beamstop_wire_bend,
        "angle": beamstop_angle, "antialias": 3, "seed": 4,
    },
)
detector = sim.DetectorConfig(
    shape=detector_shape, pixel_size=detector_pixel_size,
    sample_to_detector_distance=sample_detector_distance,
    detector_center=detector_center, beamstop_config=beamstop,
    detector_params={
        "readout_noise_average": readout_noise_average,
        "readout_noise_sigma": readout_noise_sigma,
        "detector_threshold": detector_threshold,
        "counts_per_photon": counts_per_photon,
        "quantum_efficiency": quantum_efficiency,
        "noise_seed": detector_noise_seed,
    },
    artifacts_config={
        "sigma_photon": sigma_photon,
        "photon_kernel_size": photon_kernel_size,
        "photon_n_classes": photon_n_classes,
        "photon_n_variants": photon_n_variants,
        "photon_irregularity": photon_irregularity,
        "regenerate_photon_kernels": regenerate_photon_kernels,
        "photon_kernel_seed": photon_kernel_seed,
        "camera_seed": camera_seed,
        "average_hot_pixels": average_hot_pixels,
        "average_cold_pixels": average_cold_pixels,
        "flicker_fraction": flicker_fraction,
        "flicker_probability": flicker_probability,
        "hot_pixel_value": hot_pixel_value,
        "hot_pixel_value_spread": hot_pixel_value_spread,
        "hot_pixel_temporal_sigma": hot_pixel_temporal_sigma,
        "cold_pixel_value": cold_pixel_value,
        "cold_pixel_value_spread": cold_pixel_value_spread,
        "cold_pixel_temporal_sigma": cold_pixel_temporal_sigma,
        "cosmic_rays_per_second": cosmic_rays_per_second,
        "cosmic_ray_value": cosmic_ray_value,
        "cosmic_ray_value_spread": cosmic_ray_value_spread,
        "cosmic_ray_length_range": cosmic_ray_length_range,
        "cosmic_ray_aspect_ratio_range": cosmic_ray_aspect_ratio_range,
    },
    measurement_config={
        "exposure_time": exposure_time,
        "number_frames": number_frames,
        # The API calls each frame an image, so this is the per-frame cap.
        "max_counts_per_image": max_counts_per_frame,
    },
)
detector.setup()
detector.visualize_beamstop()
real_space_pixel_size = detector.calc_realspace_resolution(xray.beam_params) / oversampling
sample_shape = [0, oversampling * detector_shape[0], oversampling * detector_shape[1]]
print(f"detector: {detector_shape}, distance: {sample_detector_distance:.3f} m")
print(f"acquisition: {number_frames} frames, maximum {max_counts_per_frame:,} counts per frame")
print(f"sample grid: {sample_shape[1:]}, pixel: {real_space_pixel_size * 1e9:.2f} nm")
print(f"coherence: {np.asarray(transverse_coherence_length) * 1e6} um")
print(f"photon kernel: sigma={sigma_photon} px, size={photon_kernel_size} px")
print(f"camera defects: mean hot={average_hot_pixels}, mean cold={average_cold_pixels}")
print(f"cosmic-ray rate: {cosmic_rays_per_second}/s; camera seed: {camera_seed}")


## 3. Define the sample geometry

The recipe defines the material stack. A binary-domain generator supplies the magnetic pattern, and the FTH mask opens one object hole (OH) and two reference holes (RH). All lengths in configuration objects are physical SI values.


In [ ]:
# ########################################
# MATERIAL STACK
# ########################################
sample = sim.SampleConfig(
    recipe="Au(1400)/SiN(50)/Pt(4)Co(20)Pt(2)",
    sample_shape=sample_shape, real_space_pixel_size=real_space_pixel_size,
    xray_config=xray, sample_name="end-to-end magnetic FTH tutorial",
)
sample.setup()
sample.sample_structure.visualize_structure()
print("layers:", sample.sample_structure.layer_names)
print("thicknesses (nm):", np.asarray(sample.sample_structure.layer_thicknesses) * 1e9)

# ########################################
# MAGNETIC PATTERN
# ########################################
magnetic = sim.MagneticPatternConfig(
    pattern_type_method="binary_labyrinth_pattern",
    shape=tuple(sample_shape[1:]), real_space_pixel_size=real_space_pixel_size,
    pattern_config={
        "stripe_width": 300e-9, "sigma": 5e-9,
        "H": 128, "W": 128, "n_steps": 60,
        "region": "custom", "use_gpu": False, "seed": 7,
        "k0": 1.05, "eps": 0.8, "target_mean": 0.0,
        "noise_amp": 0.0, "quadratic_coefficient": 0.0,
        "max_hole_area": 9, "saturation_fraction_threshold": 0.01,
    },
)
magnetic.create_pattern()
mz = magnetic.magnetic_pattern
magnetization = pattern_generator.map_magnetization_to_3d(
    magnetic_pattern_x=np.zeros_like(mz),
    magnetic_pattern_y=np.sqrt(np.clip(1.0 - mz**2, 0.0, 1.0)),
    magnetic_pattern_z=mz,
    nr_repeats=sample.sample_structure.sample_shape[0],
)
sample.assign_magnetic_pattern(magnetization)


In [ ]:
# ########################################
# HOLOGRAPHY MASK: OBJECT AND REFERENCE HOLES
# ########################################
thicknesses = sample.sample_structure.layer_thicknesses
membrane_index = sample.sample_structure.layer_names.index("SiN")
aperture = sim.FrontApertureConfig(
    aperture_method="FTH_circular", aperture_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=thicknesses,
    aperture_layer_names=sample.sample_structure.layer_names,
    aperture_config={
        "apertures_type": ["OH", "RH", "RH"],
        "apertures_radius": [0.5e-6, 60e-9, 30e-9],
        "apertures_center": [(0.0, 0.0), (1.1e-6, -1.1e-6), (-1.1e-6, -1.1e-6)],
        "apertures_sigma": [4e-9, 2e-9, 2e-9],
        "apertures_angle": [0.0, 0.0, 0.0],
        "apertures_ellipticity": [1.0, 1.0, 1.0],
        "apertures_roughness": [0.0, 0.0, 0.0],
        "apertures_roughness_modes": [(0, 0), (0, 0), (0, 0)],
        "apertures_seed": [1, 2, 3],
        "apertures_top_radius_factor": [1.0, 1.0, 1.0],
        "aperture_taper_depth": 0.0,
        "thickness_OH": float(np.sum(thicknesses[:membrane_index])),
    },
    use_roi=True,
)
aperture.setup()
aperture.visualize_aperture()
aperture_mask = aperture.return_aperture()
sample.assign_aperture_mask(aperture_mask)
sample.sample_structure.calculate_final_dielectric_tensor(use_aperture_roi=True, compact=True)

mask_projection = np.max(aperture_mask, axis=0)
extent_um = np.array([-sample_shape[2]/2, sample_shape[2]/2, sample_shape[1]/2, -sample_shape[1]/2]) * real_space_pixel_size * 1e6
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
axes[0].imshow(mz, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um); axes[0].set_title("magnetic pattern")
axes[1].imshow(mask_projection, cmap="gray", extent=extent_um); axes[1].set_title("OH + RH mask")
axes[2].imshow(mz * mask_projection, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um); axes[2].set_title("visible domains")
for ax in axes: ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")


## 4. Build the illumination on the resolved sample grid

The source and coherence parameters were chosen first. Now that detector geometry has fixed the sample grid, build the spatial illumination. `update_polarization` switches its Jones vector between the matched CR/CL helicities without rebuilding the scalar field.


In [ ]:
# ########################################
# SPATIAL ILLUMINATION ON THE SAMPLE GRID
# ########################################
illumination = sim.IlluminationConfig(
    XRayConfig=xray, shape=tuple(sample_shape[1:]),
    real_space_pixel_size=real_space_pixel_size,
    illumination_function=illumination_function,
    illumination_config={
        "center": illumination_center,
        "distance": illumination_focus_distance,
        "fwhm": illumination_fwhm,
        "alpha_beam": illumination_alpha_beam,
    },
)
illumination.setup()
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(np.abs(illumination.illumination.illumination)**2, cmap="magma", extent=extent_um)
ax.set_title("incident Gaussian intensity"); ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")


## 5. Simulate exit waves and detector holograms

For each helicity, Jones propagation produces the complex exit wave and the ideal far-field intensity. `DetectorConfig` then projects that intensity onto the physical detector and simulates the requested acquisition, applying finite coherence, the beamstop, photon statistics, photon-event kernels, readout noise, persistent hot/cold pixels, flicker, transient cosmic rays, detector threshold, and the per-frame count ceiling. The current detector API returns the frame-averaged corrupted hologram; increasing `number_frames` reduces its statistical noise rather than returning a frame stack.


In [ ]:
# ########################################
# CR / CL PROPAGATION AND DETECTION
# ########################################
results = {}
for exposure_index, helicity in enumerate(polarizations):
    illumination.update_polarization(helicity)
    propagation = sim.SamplePropagatorConfig(
        SampleConfig=sample, IlluminationConfig=illumination,
        propagator_method="Jones",
        propagator_config={
            "propagate": False, "jones_apply_zero_order_phase": True,
            "dielectric_tensor_use_roi": True,
        },
    )
    propagation.setup()
    detector.detector_params["noise_seed"] = detector_noise_seed + exposure_index
    detector.assign_propagated_wavefront(propagation)
    detector.detect_hologram()
    detected_average = detector.return_detected_hologram().copy()
    results[helicity] = {
        "exit": propagation.return_scalar_wavefield().copy(),
        "ideal": detector.return_ideal_hologram().copy(),
        "detected": detected_average,
    }
print("simulated:", list(results))
print("frame-averaged corrupted hologram shape:", results["CR"]["detected"].shape)
print("partial-coherence Gaussian sigma (y, x) in detector pixels:",
      (detector.hologram_exp.sigma_y, detector.hologram_exp.sigma_x))
print("photon-event spreading: sigma=", sigma_photon, "px; kernel size=", photon_kernel_size)


## 6. Inspect the CR and CL exit waves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for row, helicity in enumerate(("CR", "CL")):
    field = results[helicity]["exit"]
    amp = axes[row, 0].imshow(np.abs(field), cmap="magma", extent=extent_um)
    phase = axes[row, 1].imshow(np.angle(field), cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent_um)
    axes[row, 0].set_title(f"{helicity} exit amplitude")
    axes[row, 1].set_title(f"{helicity} exit phase")
    fig.colorbar(amp, ax=axes[row, 0], shrink=0.75); fig.colorbar(phase, ax=axes[row, 1], shrink=0.75)


## 7. Ideal and corrupted holograms: sum and difference

The sum `CR + CL` emphasizes nonmagnetic/charge scattering. The difference `CR - CL` isolates helicity-dependent magnetic contrast. The same algebra is applied to the ideal and corrupted detector images.


In [ ]:
holograms = {}
for source in ("ideal", "detected"):
    cr, cl = results["CR"][source], results["CL"][source]
    holograms[source] = {"CR": cr, "CL": cl, "sum": cr + cl, "difference": cr - cl}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, source in enumerate(("ideal", "detected")):
    for col, channel in enumerate(("CR", "CL", "sum", "difference")):
        data = holograms[source][channel]
        if channel == "difference":
            limit = max(np.max(np.abs(data)), np.finfo(float).eps)
            image = axes[row, col].imshow(data, cmap="RdBu_r", vmin=-limit, vmax=limit)
        else:
            floor = max(np.max(data) * 1e-8, np.finfo(float).tiny)
            image = axes[row, col].imshow(np.log10(np.maximum(data, floor)), cmap="magma")
        axes[row, col].set_title(f"{source} {channel}"); axes[row, col].set_axis_off()
        fig.colorbar(image, ax=axes[row, col], shrink=0.72)


## 8. Reconstruct sums and differences

An FTH reconstruction is the centered Fourier transform of the detector hologram. Reconstructions contain displaced object images around each reference-hole correlation peak. Use the same display scale within each channel to compare ideal and corrupted data fairly.


In [ ]:
def fth_reconstruct(hologram):
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(hologram)))

reconstructions = {
    source: {channel: fth_reconstruct(holograms[source][channel]) for channel in ("sum", "difference")}
    for source in ("ideal", "detected")
}
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, source in enumerate(("ideal", "detected")):
    for col, channel in enumerate(("sum", "difference")):
        magnitude = np.abs(reconstructions[source][channel])
        vmax = np.percentile(magnitude, 90.)
        image = axes[row, col].imshow(magnitude, cmap="inferno", vmin=0, vmax=vmax)
        axes[row, col].set_title(f"{source} |FTH({channel})|"); axes[row, col].set_axis_off()
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)


## Interpretation

- CR and CL share the structural scattering but interact oppositely with out-of-plane magnetization.
- Their sum is dominated by charge/structural contrast; their difference emphasizes magnetic circular contrast.
- The printed partial-coherence sigma distinguishes coherence blur from photon-event spreading. With a large coherence length it should approach zero pixels.
- Set `sigma_photon=0.0` to test the detector without photon kernels; `photon_kernel_size` is only the support window, while `sigma_photon` primarily controls event width.
- Beamstop `sigma` smooths the beamstop edge in physical detector coordinates and can also create a broad transition when chosen very large.
- The beamstop, finite photon statistics, detector response, and readout noise make the detected holograms more realistic and propagate into their reconstructions.
- Keep matched acquisition conditions for CR and CL; otherwise the difference also contains exposure or normalization mismatch.
- Reuse `camera_seed` for images from one camera; use distinct `noise_seed` values for independent transient draws.
